In [1]:
# OTTO Recommender System – Strong Co-Visitation Baseline
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from pathlib import Path
from tqdm.auto import tqdm
import json, gc, time, csv, sys

# Config
DATA_DIR   = Path("/kaggle/input/competitions/otto-recommender-system")
TRAIN_PATH = DATA_DIR / "train.jsonl"
TEST_PATH  = DATA_DIR / "test.jsonl"

In [2]:
N_KEEP     = 40               # how many top candidates from covis we keep per aid
N_FINAL    = 20               # final prediction length

# Weights how much each covis matrix contributes to each target
#               clicks    carts     orders
W_CLICK2CLICK = [1.00,    0.40,     0.10 ]
W_BUY2BUY     = [0.15,    1.00,     1.40 ]
W_CLICK2BUY   = [0.25,    0.80,     1.10 ]

# Recency boost for items inside the session (most recent = highest boost)
RECENCY_BOOST = [1.0, 0.8, 0.6, 0.45, 0.3, 0.2, 0.15, 0.1] + [0.07]*20

# Popularity top-up
POP_TOPN      = 40

print("Config:", {k:v for k,v in globals().items() if k.isupper() and "_" not in k[:2]})

Config: {'DATA_DIR': PosixPath('/kaggle/input/competitions/otto-recommender-system'), 'TRAIN_PATH': PosixPath('/kaggle/input/competitions/otto-recommender-system/train.jsonl'), 'TEST_PATH': PosixPath('/kaggle/input/competitions/otto-recommender-system/test.jsonl'), 'RECENCY_BOOST': [1.0, 0.8, 0.6, 0.45, 0.3, 0.2, 0.15, 0.1, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07], 'POP_TOPN': 40}


In [3]:
def load_session(line):
    d = json.loads(line)
    return int(d["session"]), [(int(e["aid"]), int(e["ts"]), e["type"]) for e in d["events"]]

In [4]:
def build_covis_matrices(path, max_lines=None):
    t0 = time.time()

    type2int = {"clicks":0, "carts":1, "orders":2}
    int2type = {v:k for k,v in type2int.items()}

    click2click = defaultdict(Counter)
    buy2buy     = defaultdict(Counter)
    click2buy   = defaultdict(Counter)   # from click to later cart/order

    popular     = Counter()              # global popularity (mostly for fallback)

    n = 0
    with open(path, "r") as f:
        for line in tqdm(f, desc="Build covis", total=12_887_413 if "train" in str(path) else 1_671_803):
            sid, events = load_session(line)
            if not events: continue

            # take tail of long sessions
            events = events[-20:]

            aids = [aid for aid,_,_ in events]
            types = [type2int[t] for _,_,t in events]

            # global popularity
            for a in aids:
                popular[a] += 1

            for i in range(len(events)):
                if types[i] != 0: continue
                a = aids[i]
                for j in range(i+1, min(i+35, len(events))):
                    if types[j] != 0: continue
                    b = aids[j]
                    click2click[a][b] += 1
                    click2click[b][a] += 1   # symmetric

            buy_aids = set()
            for aid, typ in zip(aids, types):
                if typ >= 1:
                    buy_aids.add(aid)

            buy_list = list(buy_aids)
            for i in range(len(buy_list)):
                for j in range(i+1, len(buy_list)):
                    a, b = buy_list[i], buy_list[j]
                    buy2buy[a][b] += 1
                    buy2buy[b][a] += 1

            for i in range(len(events)):
                if types[i] != 0: continue
                a = aids[i]
                for j in range(i+1, len(events)):
                    if types[j] == 0: continue
                    b = aids[j]
                    click2buy[a][b] += 1

            n += 1
            if max_lines is not None and n >= max_lines:
                break

    # Keep only top N neighbors
    for d in [click2click, buy2buy, click2buy]:
        for aid in list(d):
            if len(d[aid]) > N_KEEP*2:
                top = d[aid].most_common(N_KEEP*2)
                d[aid] = Counter(dict(top))

    print(f"Built {n:,} sessions in {time.time()-t0:.1f} s")
    print("click2click aids:", len(click2click))
    print("buy2buy     aids:", len(buy2buy))
    print("click2buy   aids:", len(click2buy))
    print("Popular items  :", len(popular))

    return click2click, buy2buy, click2buy, popular.most_common(POP_TOPN)

In [5]:
print("\nBuilding matrices from train...")
click2click, buy2buy, click2buy, pop_items = build_covis_matrices(TRAIN_PATH)


Building matrices from train...


Build covis:   0%|          | 0/12887413 [00:00<?, ?it/s]

Built 12,899,779 sessions in 1479.3 s
click2click aids: 1830613
buy2buy     aids: 868578
click2buy   aids: 1477922
Popular items  : 1832220


In [6]:
def recommend(session_events, target_type):
    # target_type: "clicks", "carts", "orders"

    if target_type == "clicks":    tgt = 0
    elif target_type == "carts":   tgt = 1
    else:                          tgt = 2

    w_c2c = W_CLICK2CLICK[tgt]
    w_b2b = W_BUY2BUY[tgt]
    w_c2b = W_CLICK2BUY[tgt]

    aids = [aid for aid,_,typ in session_events if typ == "clicks" or tgt > 0]
    if not aids:
        return [aid for aid,_ in pop_items][:N_FINAL]

    # reverse = most recent first
    aids = aids[::-1]
    unique_recent = []
    seen = set()
    for a in aids:
        if a not in seen:
            unique_recent.append(a)
            seen.add(a)
        if len(unique_recent) >= 35: break

    scores = Counter()

    # recency bias
    for pos, a in enumerate(unique_recent):
        boost = RECENCY_BOOST[min(pos, len(RECENCY_BOOST)-1)]
        scores[a] += boost * 2.5   # self-boost

        # click2click neighbors
        for nb, cnt in click2click[a].items():
            scores[nb] += cnt * w_c2c

        # buy2buy neighbors
        for nb, cnt in buy2buy[a].items():
            scores[nb] += cnt * w_b2b

        # click2buy neighbors
        for nb, cnt in click2buy[a].items():
            scores[nb] += cnt * w_c2b

    # remove already seen
    for a in unique_recent:
        scores[a] = -9999

    # top candidates
    cands = [aid for aid,sc in scores.most_common(N_KEEP*3)]

    # fill with pure popularity if needed
    rec = []
    used = set(unique_recent)

    for a in cands:
        if a not in used:
            rec.append(a)
            used.add(a)
        if len(rec) >= N_FINAL: break

    for aid, _ in pop_items:
        if aid not in used:
            rec.append(aid)
            used.add(aid)
        if len(rec) >= N_FINAL: break

    return rec[:N_FINAL]

In [7]:
# Submit
with open(TEST_PATH, "r") as f, open("submission.csv", "w", newline="") as out:
    writer = csv.writer(out)
    writer.writerow(["session_type", "labels"])

    for line in tqdm(f, desc="Inference", total=1_671_803):
        sid, events = load_session(line)

        for target in ["clicks", "carts", "orders"]:
            preds = recommend(events, target)
            labels_str = " ".join(map(str, preds))
            writer.writerow([f"{sid}_{target}", labels_str])

Inference:   0%|          | 0/1671803 [00:00<?, ?it/s]